# Explore `dataset_final.jsonl`

Streams the JSONL file line by line. The file is ~566 MB, so do **not** load it all at once.

In [1]:
import json
from pathlib import Path

DATASET_PATH = Path("dataset_final.jsonl")
DATASET_PATH.exists(), DATASET_PATH.stat().st_size / 1e6

(True, 593.22644)

In [2]:
def iter_records(path=DATASET_PATH):
    """Yield one parsed JSON object per line."""
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

In [3]:
first = next(iter_records())
print("top-level keys:", list(first.keys()))
print("name:", first.get("name"))
print("smiles:", first.get("smiles"))
print("num qa_pairs:", len(first.get("qa_pairs", [])))
print("num evidence_sentences:", len(first.get("evidence_sentences", [])))

top-level keys: ['cid', 'split', 'name', 'iupac_name', 'smiles', 'molecular_formula', 'molecular_weight', 'inchi_key', 'num_pmids', 'num_synonyms', 'num_evidence_sentences', 'evidence_sentences', 'qa_pairs']
name: 1-Amino-2-propanol
smiles: CC(CN)O
num qa_pairs: 7
num evidence_sentences: 1


In [4]:
# Peek at the first N records without holding the whole file in memory.
from itertools import islice

for rec in islice(iter_records(), 3):
    print(rec["cid"], rec["split"], rec["name"])

4 train 1-Amino-2-propanol
6 train 1-Chloro-2,4-Dinitrobenzene
11 train 1,2-Dichloroethane


In [5]:
# Count records and split distribution by streaming once.
from collections import Counter

splits = Counter()
n = 0
for rec in iter_records():
    splits[rec.get("split")] += 1
    n += 1
print("total records:", n)
print("by split:", dict(splits))

total records: 15509
by split: {'train': 10820, 'val': 2340, 'test': 2349}


In [6]:
# Disagree rate per topic bucket: disagree / (agree + disagree)
# Buckets: structural / functional / engineering / other.
# `engineering` is normally folded into `functional` by topic_bucket.py;
# we split it out here so the design-leverage QAs can be inspected separately.
from collections import Counter
from topic_bucket import bucket_topic

ENGINEERING_KEYS = {"engineering", "design_levers", "design"}
ENGINEERING_SUBS = ("engineer", "design")

def bucket4(topic):
    t = (topic or "").strip().lower().replace("-", "_")
    if t in ENGINEERING_KEYS or any(s in t for s in ENGINEERING_SUBS):
        return "engineering"
    return bucket_topic(topic)

BUCKETS = ("structural", "functional", "engineering", "other")
counts = {b: Counter() for b in BUCKETS}

for rec in iter_records():
    for qa in rec.get("qa_pairs", []):
        counts[bucket4(qa.get("topic"))][qa.get("verdict")] += 1

print(f"{'bucket':<12} {'agree':>8} {'disagree':>9} {'a+d':>8} {'disagree_rate':>14}")
for b in BUCKETS:
    a = counts[b].get("agree", 0)
    d = counts[b].get("disagree", 0)
    denom = a + d
    rate = d / denom if denom else float("nan")
    print(f"{b:<12} {a:>8} {d:>9} {denom:>8} {rate:>14.4f}")

print("\nfull verdict distribution per bucket:")
for b in BUCKETS:
    print(f"  {b}: {dict(counts[b])}")

bucket          agree  disagree      a+d  disagree_rate
structural      80427     17788    98215         0.1811
functional      70812      1384    72196         0.0192
engineering     26874      1864    28738         0.0649
other           10428       128    10556         0.0121

full verdict distribution per bucket:
  structural: {'agree': 80427, 'disagree': 17788, None: 294, 'unclear': 8}
  functional: {'agree': 70812, 'disagree': 1384, None: 50, 'unclear': 19}
  engineering: {'agree': 26874, 'disagree': 1864, None: 31, 'unclear': 4}
  other: {'agree': 10428, None: 6, 'disagree': 128}


In [7]:
# Per-topic disagree rate: disagree_count / (agree + disagree)
# Buckets: structural / functional / engineering / other.
from collections import Counter
from topic_bucket import bucket_topic

ENGINEERING_KEYS = {"engineering", "design_levers", "design"}
ENGINEERING_SUBS = ("engineer", "design")

def bucket4(topic):
    t = (topic or "").strip().lower().replace("-", "_")
    if t in ENGINEERING_KEYS or any(s in t for s in ENGINEERING_SUBS):
        return "engineering"
    return bucket_topic(topic)

BUCKETS = ("structural", "functional", "engineering", "other")
counts = {b: Counter() for b in BUCKETS}

for rec in iter_records():
    for qa in rec.get("qa_pairs", []):
        counts[bucket4(qa.get("topic"))][qa.get("verdict")] += 1

print(f"{'bucket':<12} {'agree':>8} {'disagree':>9} {'a+d':>8} {'disagree_rate':>14}")
for b in BUCKETS:
    a = counts[b].get("agree", 0)
    d = counts[b].get("disagree", 0)
    denom = a + d
    rate = d / denom if denom else float("nan")
    print(f"{b:<12} {a:>8} {d:>9} {denom:>8} {rate:>14.4f}")

print("\nfull verdict distribution per bucket:")
for b in BUCKETS:
    print(f"  {b}: {dict(counts[b])}")

bucket          agree  disagree      a+d  disagree_rate
structural      80427     17788    98215         0.1811
functional      70812      1384    72196         0.0192
engineering     26874      1864    28738         0.0649
other           10428       128    10556         0.0121

full verdict distribution per bucket:
  structural: {'agree': 80427, 'disagree': 17788, None: 294, 'unclear': 8}
  functional: {'agree': 70812, 'disagree': 1384, None: 50, 'unclear': 19}
  engineering: {'agree': 26874, 'disagree': 1864, None: 31, 'unclear': 4}
  other: {'agree': 10428, None: 6, 'disagree': 128}


In [8]:
# Tables of counts across the JSONL — single streaming pass.
from collections import Counter
import statistics as stats
from topic_bucket import bucket_topic

split_counts = Counter()
qa_per_split = Counter()
verdict_counts = Counter()
verdict_by_split = {}
topic_counts_raw = Counter()
topic_bucket_counts = Counter()
verdict_by_bucket = {}
qa_per_record = []
evidence_per_record = []
pmids_per_record = []
synonyms_per_record = []
mw_values = []
has_iupac = 0
has_formula = 0
has_inchi = 0
total_records = 0
total_qas = 0
total_evidence = 0

for rec in iter_records():
    total_records += 1
    split = rec.get("split")
    split_counts[split] += 1
    verdict_by_split.setdefault(split, Counter())

    qas = rec.get("qa_pairs", []) or []
    evid = rec.get("evidence_sentences", []) or []
    qa_per_record.append(len(qas))
    evidence_per_record.append(len(evid))
    qa_per_split[split] += len(qas)
    total_qas += len(qas)
    total_evidence += len(evid)

    if rec.get("num_pmids") is not None:
        pmids_per_record.append(rec["num_pmids"])
    if rec.get("num_synonyms") is not None:
        synonyms_per_record.append(rec["num_synonyms"])
    mw = rec.get("molecular_weight")
    if isinstance(mw, (int, float)):
        mw_values.append(mw)
    if rec.get("iupac_name"):
        has_iupac += 1
    if rec.get("molecular_formula"):
        has_formula += 1
    if rec.get("inchi_key"):
        has_inchi += 1

    for qa in qas:
        v = qa.get("verdict")
        t = qa.get("topic")
        verdict_counts[v] += 1
        verdict_by_split[split][v] += 1
        topic_counts_raw[t] += 1
        topic_bucket_counts[bucket_topic(t)] += 1
        verdict_by_bucket.setdefault(bucket_topic(t), Counter())[v] += 1

def describe(xs):
    if not xs:
        return "n=0"
    return (f"n={len(xs)} min={min(xs)} max={max(xs)} "
            f"mean={stats.mean(xs):.2f} median={stats.median(xs)}")

print("=" * 70)
print(f"TOTAL RECORDS: {total_records:,}   TOTAL QA PAIRS: {total_qas:,}   "
      f"TOTAL EVIDENCE SENTENCES: {total_evidence:,}")
print("=" * 70)

print("\n[1] Records by split")
print(f"{'split':<8} {'records':>10} {'qa_pairs':>10} {'qa/record':>11}")
for s, n in split_counts.most_common():
    avg = qa_per_split[s] / n if n else 0
    print(f"{str(s):<8} {n:>10,} {qa_per_split[s]:>10,} {avg:>11.2f}")

print("\n[2] QA verdict distribution (overall)")
print(f"{'verdict':<10} {'count':>10} {'pct':>8}")
tot_v = sum(verdict_counts.values())
for v, c in verdict_counts.most_common():
    print(f"{str(v):<10} {c:>10,} {c/tot_v*100:>7.2f}%")

print("\n[3] QA verdict by split")
header = f"{'split':<8}" + "".join(f"{str(v):>10}" for v in verdict_counts)
print(header)
for s in split_counts:
    row = f"{str(s):<8}" + "".join(f"{verdict_by_split[s].get(v,0):>10,}" for v in verdict_counts)
    print(row)

print("\n[4] QA topic — bucketed (structural/functional/other via topic_bucket.py)")
print(f"{'bucket':<12} {'count':>10} {'pct':>8}")
tot_b = sum(topic_bucket_counts.values())
for b, c in topic_bucket_counts.most_common():
    print(f"{b:<12} {c:>10,} {c/tot_b*100:>7.2f}%")

print("\n[5] QA topic — raw labels (top 25)")
print(f"{'topic':<35} {'count':>10}")
for t, c in topic_counts_raw.most_common(25):
    print(f"{str(t):<35} {c:>10,}")
print(f"... {len(topic_counts_raw)} unique topic strings total")

print("\n[6] Verdict × bucket")
verds = list(verdict_counts.keys())
header = f"{'bucket':<12}" + "".join(f"{str(v):>10}" for v in verds)
print(header)
for b in topic_bucket_counts:
    row = f"{b:<12}" + "".join(f"{verdict_by_bucket[b].get(v,0):>10,}" for v in verds)
    print(row)

print("\n[7] Per-record distributions")
print(f"  qa_pairs/record:       {describe(qa_per_record)}")
print(f"  evidence/record:       {describe(evidence_per_record)}")
print(f"  num_pmids/record:      {describe(pmids_per_record)}")
print(f"  num_synonyms/record:   {describe(synonyms_per_record)}")
print(f"  molecular_weight:      {describe(mw_values)}")

print("\n[8] Field presence (records with non-empty value)")
print(f"  iupac_name:            {has_iupac:>6,} / {total_records:,} ({has_iupac/total_records*100:.1f}%)")
print(f"  molecular_formula:     {has_formula:>6,} / {total_records:,} ({has_formula/total_records*100:.1f}%)")
print(f"  inchi_key:             {has_inchi:>6,} / {total_records:,} ({has_inchi/total_records*100:.1f}%)")

TOTAL RECORDS: 15,509   TOTAL QA PAIRS: 210,117   TOTAL EVIDENCE SENTENCES: 1,101,940

[1] Records by split
split       records   qa_pairs   qa/record
train        10,820    149,756       13.84
test          2,349     29,635       12.62
val           2,340     30,726       13.13

[2] QA verdict distribution (overall)
verdict         count      pct
agree         188,541   89.73%
disagree       21,164   10.07%
None              381    0.18%
unclear            31    0.01%

[3] QA verdict by split
split        agree  disagree      None   unclear
train      135,234    14,218       283        21
val         27,102     3,560        58         6
test        26,205     3,386        40         4

[4] QA topic — bucketed (structural/functional/other via topic_bucket.py)
bucket            count      pct
functional      101,037   48.09%
structural       98,518   46.89%
other            10,562    5.03%

[5] QA topic — raw labels (top 25)
topic                                    count
engineering    

In [9]:
# Agree vs disagree variations across categories/criteria.
# Restrict to verdict in {"agree","disagree"}; ignore None/unclear.
from collections import Counter, defaultdict
from topic_bucket import bucket_topic

ENGINEERING_KEYS = {"engineering", "design_levers", "design"}
ENGINEERING_SUBS = ("engineer", "design")
def bucket4(topic):
    t = (topic or "").strip().lower().replace("-", "_")
    if t in ENGINEERING_KEYS or any(s in t for s in ENGINEERING_SUBS):
        return "engineering"
    return bucket_topic(topic)

# slice-name -> key -> [agree, disagree]
slices = defaultdict(lambda: defaultdict(lambda: [0, 0]))

def mw_bin(mw):
    if mw is None: return "unknown"
    for lo, hi, lbl in [(0,150,"<150"),(150,300,"150-300"),(300,500,"300-500"),
                        (500,900,"500-900"),(900,float("inf"),">=900")]:
        if lo <= mw < hi: return lbl
    return "unknown"

def bin_int(x, edges, labels):
    for e, l in zip(edges, labels):
        if x <= e: return l
    return labels[-1]

PMID_EDGES = [1,3,10,30,100]; PMID_LABELS = ["1","2-3","4-10","11-30","31-100",">100"]
SYN_EDGES  = [10,50,200,500]; SYN_LABELS  = ["<=10","11-50","51-200","201-500",">500"]
EV_EDGES   = [3,10,50,200];   EV_LABELS   = ["1-3","4-10","11-50","51-200",">200"]
QA_EDGES   = [7,10,15,25];    QA_LABELS   = ["<=7","8-10","11-15","16-25",">25"]

for rec in iter_records():
    split = rec.get("split")
    mw = rec.get("molecular_weight") if isinstance(rec.get("molecular_weight"),(int,float)) else None
    mwb = mw_bin(mw)
    pmb = bin_int(rec.get("num_pmids",0), PMID_EDGES, PMID_LABELS)
    syb = bin_int(rec.get("num_synonyms",0), SYN_EDGES, SYN_LABELS)
    evb = bin_int(rec.get("num_evidence_sentences",0), EV_EDGES, EV_LABELS)
    qab = bin_int(len(rec.get("qa_pairs") or []), QA_EDGES, QA_LABELS)
    for qa in rec.get("qa_pairs") or []:
        v = qa.get("verdict")
        if v not in ("agree","disagree"): continue
        idx = 0 if v == "agree" else 1
        slices["split"][split][idx] += 1
        slices["bucket3"][bucket_topic(qa.get("topic"))][idx] += 1
        slices["bucket4"][bucket4(qa.get("topic"))][idx] += 1
        slices["topic_raw"][qa.get("topic")][idx] += 1
        slices["mw_bin"][mwb][idx] += 1
        slices["pmids_bin"][pmb][idx] += 1
        slices["synonyms_bin"][syb][idx] += 1
        slices["evidence_bin"][evb][idx] += 1
        slices["qa_per_record_bin"][qab][idx] += 1

def print_slice(name, data, sort="key", order=None, top=None, min_n=0):
    rows = []
    for k,(a,d) in data.items():
        n = a + d
        if n < min_n: continue
        rows.append((k, a, d, n, d/n if n else 0))
    if order:
        rows.sort(key=lambda r: order.index(r[0]) if r[0] in order else 1e9)
    elif sort == "rate":
        rows.sort(key=lambda r: -r[4])
    elif sort == "n":
        rows.sort(key=lambda r: -r[3])
    else:
        rows.sort(key=lambda r: str(r[0]))
    if top: rows = rows[:top]
    print(f"\n[{name}]")
    print(f"  {'key':<28} {'agree':>9} {'disagree':>9} {'n':>9} {'disagree%':>10} {'agree%':>8}")
    for k,a,d,n,r in rows:
        print(f"  {str(k):<28} {a:>9,} {d:>9,} {n:>9,} {r*100:>9.2f}% {(1-r)*100:>7.2f}%")

grand = [0,0]
for a,d in slices["split"].values():
    grand[0]+=a; grand[1]+=d
g_n = grand[0]+grand[1]
print(f"OVERALL (agree+disagree only): n={g_n:,}  "
      f"agree={grand[0]:,} ({grand[0]/g_n*100:.2f}%)  "
      f"disagree={grand[1]:,} ({grand[1]/g_n*100:.2f}%)")

print_slice("by split", slices["split"], order=["train","val","test"])
print_slice("by topic bucket (3-way, engineering folded into functional)",
            slices["bucket3"], sort="rate")
print_slice("by topic bucket (4-way, engineering split out)",
            slices["bucket4"], sort="rate")
print_slice("by molecular_weight bin", slices["mw_bin"],
            order=["<150","150-300","300-500","500-900",">=900","unknown"])
print_slice("by num_pmids bin", slices["pmids_bin"], order=PMID_LABELS)
print_slice("by num_synonyms bin", slices["synonyms_bin"], order=SYN_LABELS)
print_slice("by num_evidence_sentences bin", slices["evidence_bin"], order=EV_LABELS)
print_slice("by qa_pairs-per-record bin", slices["qa_per_record_bin"], order=QA_LABELS)
print_slice("top 15 raw topics by disagree% (min n=500)",
            slices["topic_raw"], sort="rate", top=15, min_n=500)
print_slice("bottom 15 raw topics by disagree% (min n=500)",
            {k:v for k,v in sorted(slices["topic_raw"].items(),
                                   key=lambda kv:(kv[1][1]/(kv[1][0]+kv[1][1]) if sum(kv[1]) else 0))[:15]
             if sum(v)>=500}, sort="rate")

OVERALL (agree+disagree only): n=209,705  agree=188,541 (89.91%)  disagree=21,164 (10.09%)

[by split]
  key                              agree  disagree         n  disagree%   agree%
  train                          135,234    14,218   149,452      9.51%   90.49%
  val                             27,102     3,560    30,662     11.61%   88.39%
  test                            26,205     3,386    29,591     11.44%   88.56%

[by topic bucket (3-way, engineering folded into functional)]
  key                              agree  disagree         n  disagree%   agree%
  structural                      80,428    17,788    98,216     18.11%   81.89%
  functional                      97,685     3,248   100,933      3.22%   96.78%
  other                           10,428       128    10,556      1.21%   98.79%

[by topic bucket (4-way, engineering split out)]
  key                              agree  disagree         n  disagree%   agree%
  structural                      80,427    17,788    9

Variations observed (disagree% is what moves — agree% is its complement)
Split — train 9.51% vs val 11.61% / test 11.44%. Val+test are ~2pp higher than train, likely because rarer/harder molecules are oversampled in held-out splits.

Topic bucket (3-way) — structural 18.11% · functional 3.22% · other 1.21%. Structural is ~6× functional.

Topic bucket (4-way, engineering split) — structural 18.11% · engineering 6.49% · functional 1.92% · other 1.21%. Engineering is meaningfully harder than pure functional.

Raw topics (top 15 disagree%, min n=500) — heavy variation:

stereochemistry 44.16% (near-coinflip!)
scaffold 27.95% · druglikeness 27.48% · composition 22.59% · functional_groups 17.72% · shape_sterics 15.80%
Low end: drug_interactions 2.06% · adme 2.32% · metabolism 3.92%
Range within structural family is 8–44%, not uniform.
Molecular weight — non-monotonic:

<150: 4.97% · 150–300: 9.16% · 300–500: 12.65% (peak) · 500–900: 11.86% · ≥900: 9.10%
Mid-size drug-like molecules are the hardest; very small and very large easier.
Evidence-volume signal (monotonic, all three) — more data → fewer disagreements:

num_pmids: 1→15.19%, 2–3→12.28%, 4–10→10.32%, 11–30→8.11%, 31–100→7.17%, >100→6.86%
num_synonyms: ≤10→13.43% … >500→3.84%
num_evidence_sentences: 1–3→15.32% … >200→7.10%
Roughly 2× disagreement when evidence is sparse.
QA density per record — also monotonic:

≤7 QAs/rec: 14.45% · 11–15: 10.72% · 16–25: 8.51% · >25: 7.24%
Records with many QAs are "easy" molecules (plenty to say correctly).
Bottom-line
The disagree rate is not uniform — it varies by ~22× across slices. The three biggest drivers, in order:

Topic — stereochemistry (44%) vs drug_interactions (2%) is a 20× gap.
Evidence volume — sparse-evidence records disagree ~2× as often as data-rich ones (consistent across pmids / synonyms / evidence sentences).
Molecular weight — drug-like mid-range (300–500 Da) is ~2.5× harder than very small molecules.
Split and bucket variation are real but smaller-scale effects.

(Note: the "bottom 15 topics" table came up empty — my filter ordered first then cut to 15, so nothing with n≥500 survived. The top-15 table already contains the low end, so you can read the bottom from there: drug_interactions 2.06% is the floor among common topics.)

In [10]:
# Correlations between numerical fields and agree/disagree counts (stdlib only).
# Features per record: molecular_weight, num_pmids, num_synonyms,
#                      num_evidence_sentences, num_qa_pairs.
# Targets per record:  agree_count, disagree_count, disagree_rate.
# Also QA-level: disagree flag (0/1) vs each numeric feature (point-biserial).
import math
from statistics import mean

FEATS = ["molecular_weight","num_pmids","num_synonyms",
         "num_evidence_sentences","num_qa_pairs"]

def pearson(xs, ys):
    n = len(xs)
    if n < 3: return float("nan"), float("nan")
    mx, my = mean(xs), mean(ys)
    num = sum((x-mx)*(y-my) for x,y in zip(xs,ys))
    dx2 = sum((x-mx)**2 for x in xs)
    dy2 = sum((y-my)**2 for y in ys)
    if dx2 == 0 or dy2 == 0: return float("nan"), float("nan")
    r = num / math.sqrt(dx2*dy2)
    # two-sided p via t-distribution approximation
    r2 = min(max(r*r, 0.0), 1-1e-15)
    t = r * math.sqrt((n-2)/(1-r2))
    # Use survival of |t| under t(n-2) via normal approx for large n
    # (n here is 15k+ or 209k+, so normal is fine)
    p = 2 * (1 - 0.5*(1 + math.erf(abs(t)/math.sqrt(2))))
    return r, p

def ranks(values):
    # average ranks for ties
    idx = sorted(range(len(values)), key=lambda i: values[i])
    r = [0.0]*len(values)
    i = 0
    while i < len(values):
        j = i
        while j+1 < len(values) and values[idx[j+1]] == values[idx[i]]:
            j += 1
        avg = (i + j) / 2 + 1  # 1-based average
        for k in range(i, j+1):
            r[idx[k]] = avg
        i = j + 1
    return r

def spearman(xs, ys):
    return pearson(ranks(xs), ranks(ys))

# Collect record-level and qa-level data
rec_feats = [[] for _ in FEATS]
rec_agree, rec_disagree, rec_rate = [], [], []
qa_feats = [[] for _ in FEATS]
qa_is_dis = []

for rec in iter_records():
    a = d = 0
    for qa in rec.get("qa_pairs") or []:
        v = qa.get("verdict")
        if v == "agree": a += 1
        elif v == "disagree": d += 1
    if a + d == 0:
        continue
    mw = rec.get("molecular_weight")
    if not isinstance(mw,(int,float)): mw = None
    feat = [
        mw,
        rec.get("num_pmids") or 0,
        rec.get("num_synonyms") or 0,
        rec.get("num_evidence_sentences") or 0,
        len(rec.get("qa_pairs") or []),
    ]
    for i, v in enumerate(feat):
        rec_feats[i].append(v)
    rec_agree.append(a); rec_disagree.append(d); rec_rate.append(d/(a+d))
    for qa in rec.get("qa_pairs") or []:
        v = qa.get("verdict")
        if v in ("agree","disagree"):
            for i, fv in enumerate(feat):
                qa_feats[i].append(fv)
            qa_is_dis.append(1 if v == "disagree" else 0)

print(f"records used: {len(rec_agree):,}   qa pairs used: {len(qa_is_dis):,}")

def corr_table(X_cols, y, name):
    print(f"\n--- {name} ---")
    print(f"  {'feature':<26} {'Pearson r':>10} {'p(P)':>10} {'Spearman rho':>14} {'p(S)':>10}")
    for i, f in enumerate(FEATS):
        xs, ys = [], []
        for xv, yv in zip(X_cols[i], y):
            if xv is None or yv is None: continue
            xs.append(float(xv)); ys.append(float(yv))
        pr, pp = pearson(xs, ys)
        sr, sp = spearman(xs, ys)
        print(f"  {f:<26} {pr:>10.4f} {pp:>10.2e} {sr:>14.4f} {sp:>10.2e}")

corr_table(rec_feats, rec_agree,    "record-level: feature vs AGREE count")
corr_table(rec_feats, rec_disagree, "record-level: feature vs DISAGREE count")
corr_table(rec_feats, rec_rate,     "record-level: feature vs DISAGREE RATE (d/(a+d))")
corr_table(qa_feats,  qa_is_dis,    "qa-level: feature vs IS_DISAGREE (0/1)  [point-biserial]")

print("\n--- feature x feature Pearson matrix (record-level) ---")
# drop rows where MW is None
keep = [i for i,v in enumerate(rec_feats[0]) if v is not None]
cols = [[rec_feats[j][i] for i in keep] for j in range(len(FEATS))]
print("  " + " ".join(f"{f[:12]:>12}" for f in FEATS))
for i, f in enumerate(FEATS):
    row = []
    for j in range(len(FEATS)):
        r, _ = pearson([float(v) for v in cols[i]], [float(v) for v in cols[j]])
        row.append(r)
    print(f"  {f[:12]:<12} " + " ".join(f"{r:>12.3f}" for r in row))

records used: 15,509   qa pairs used: 209,705

--- record-level: feature vs AGREE count ---
  feature                     Pearson r       p(P)   Spearman rho       p(S)
  molecular_weight               0.0182   2.35e-02        -0.0450   2.00e-08
  num_pmids                      0.7189   0.00e+00         0.7894   0.00e+00
  num_synonyms                   0.4260   0.00e+00         0.3991   0.00e+00
  num_evidence_sentences         0.9117   0.00e+00         0.8494   0.00e+00
  num_qa_pairs                   0.9919   0.00e+00         0.9026   0.00e+00

--- record-level: feature vs DISAGREE count ---
  feature                     Pearson r       p(P)   Spearman rho       p(S)
  molecular_weight               0.0318   7.34e-05         0.2200   0.00e+00
  num_pmids                      0.2621   0.00e+00         0.2385   0.00e+00
  num_synonyms                   0.1099   0.00e+00         0.0873   0.00e+00


  num_evidence_sentences         0.3715   0.00e+00         0.2989   0.00e+00
  num_qa_pairs                   0.4264   0.00e+00         0.3771   0.00e+00

--- record-level: feature vs DISAGREE RATE (d/(a+d)) ---
  feature                     Pearson r       p(P)   Spearman rho       p(S)
  molecular_weight               0.0054   4.99e-01         0.2057   0.00e+00
  num_pmids                     -0.1489   0.00e+00        -0.1936   0.00e+00
  num_synonyms                  -0.1509   0.00e+00        -0.1341   0.00e+00
  num_evidence_sentences        -0.1791   0.00e+00        -0.1682   0.00e+00
  num_qa_pairs                  -0.2114   0.00e+00        -0.1136   0.00e+00

--- qa-level: feature vs IS_DISAGREE (0/1)  [point-biserial] ---
  feature                     Pearson r       p(P)   Spearman rho       p(S)


  molecular_weight               0.0047   3.00e-02         0.0699   0.00e+00


  num_pmids                     -0.0628   0.00e+00        -0.1006   0.00e+00


  num_synonyms                  -0.0532   0.00e+00        -0.0623   0.00e+00


  num_evidence_sentences        -0.0775   0.00e+00        -0.0967   0.00e+00


  num_qa_pairs                  -0.0902   0.00e+00        -0.0903   0.00e+00

--- feature x feature Pearson matrix (record-level) ---
  molecular_we    num_pmids num_synonyms num_evidence num_qa_pairs
  molecular_we        1.000        0.013       -0.155        0.021        0.022
  num_pmids           0.013        1.000        0.408        0.822        0.718
  num_synonyms       -0.155        0.408        1.000        0.406        0.419
  num_evidence        0.021        0.822        0.406        1.000        0.916


  num_qa_pairs        0.022        0.718        0.419        0.916        1.000


In [11]:
# CIDs with > 50% disagree rate: how many, and how do they differ from the rest?
# Features compared: molecular_weight, heavy_atom_count (from molecular_formula),
# num_pmids, num_synonyms, num_evidence_sentences, num_qa_pairs, split.
import math, re
from collections import Counter
from statistics import mean, median

FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\d*)")
def heavy_atom_count(formula):
    """Count non-H atoms from a molecular formula string like 'C6H12O6' or 'CH4Cl-'."""
    if not formula: return None
    total = 0
    for sym, num in FORMULA_RE.findall(formula):
        if sym == "H": continue
        total += int(num) if num else 1
    return total

def pearson(xs, ys):
    n = len(xs)
    if n < 3: return float("nan"), float("nan")
    mx, my = mean(xs), mean(ys)
    num = sum((x-mx)*(y-my) for x,y in zip(xs,ys))
    dx2 = sum((x-mx)**2 for x in xs); dy2 = sum((y-my)**2 for y in ys)
    if dx2 == 0 or dy2 == 0: return float("nan"), float("nan")
    r = num / math.sqrt(dx2*dy2)
    r2 = min(max(r*r, 0.0), 1-1e-15)
    t = r * math.sqrt((n-2)/(1-r2))
    p = 2 * (1 - 0.5*(1 + math.erf(abs(t)/math.sqrt(2))))
    return r, p

# Gather per-record data
recs = []  # dict per record
for rec in iter_records():
    a = d = 0
    for qa in rec.get("qa_pairs") or []:
        v = qa.get("verdict")
        if v == "agree": a += 1
        elif v == "disagree": d += 1
    if a + d == 0: continue
    mw = rec.get("molecular_weight")
    recs.append({
        "cid": rec.get("cid"),
        "name": rec.get("name"),
        "split": rec.get("split"),
        "mw": mw if isinstance(mw,(int,float)) else None,
        "heavy": heavy_atom_count(rec.get("molecular_formula")),
        "pmids": rec.get("num_pmids") or 0,
        "syn": rec.get("num_synonyms") or 0,
        "evid": rec.get("num_evidence_sentences") or 0,
        "nqa": len(rec.get("qa_pairs") or []),
        "agree": a, "disagree": d,
        "rate": d / (a + d),
        "n": a + d,
    })

high = [r for r in recs if r["rate"] > 0.5]
low  = [r for r in recs if r["rate"] <= 0.5]
print(f"records total: {len(recs):,}")
print(f"records with disagree-rate > 50%: {len(high):,} "
      f"({len(high)/len(recs)*100:.2f}%)")
print(f"records with disagree-rate <= 50%: {len(low):,}")

# Stratify the high group by how extreme
strata = [(0.5,0.6),(0.6,0.7),(0.7,0.8),(0.8,0.9),(0.9,1.01)]
print("\nhigh-disagree stratification:")
print(f"  {'range':<12} {'count':>6}")
for lo, hi in strata:
    c = sum(1 for r in high if lo < r["rate"] <= hi)
    print(f"  {f'{lo:.1f}<r<={hi:.2f}':<12} {c:>6}")

# 20 worst CIDs (highest disagree rate, min n=5)
print("\nTop 20 CIDs by disagree rate (min 5 agree+disagree QAs):")
print(f"  {'cid':>8} {'split':<6} {'name':<32} {'n':>4} {'dis%':>7} {'mw':>8} {'heavy':>6} {'evid':>6} {'pmids':>6}")
for r in sorted(high, key=lambda r: (-r["rate"], -r["n"]))[:20]:
    if r["n"] < 5: continue
    name = (r["name"] or "")[:32]
    mw = f"{r['mw']:.1f}" if r['mw'] is not None else "n/a"
    hv = r["heavy"] if r["heavy"] is not None else "n/a"
    print(f"  {r['cid']:>8} {str(r['split']):<6} {name:<32} {r['n']:>4} "
          f"{r['rate']*100:>6.1f}% {mw:>8} {str(hv):>6} {r['evid']:>6} {r['pmids']:>6}")

# Summary stats: high-disagree vs rest
def summarize(group, label):
    def stat(key):
        xs = [r[key] for r in group if r[key] is not None]
        if not xs: return "n/a"
        return f"n={len(xs)} mean={mean(xs):.2f} median={median(xs):.2f}"
    print(f"\n[{label}] n={len(group):,}")
    print(f"  mw:     {stat('mw')}")
    print(f"  heavy:  {stat('heavy')}")
    print(f"  pmids:  {stat('pmids')}")
    print(f"  syn:    {stat('syn')}")
    print(f"  evid:   {stat('evid')}")
    print(f"  nqa:    {stat('nqa')}")
    sc = Counter(r["split"] for r in group)
    tot = sum(sc.values())
    print(f"  split:  " + ", ".join(f"{s}={sc.get(s,0)} ({sc.get(s,0)/tot*100:.1f}%)"
                                     for s in ["train","val","test"]))

summarize(high, "disagree-rate > 50%")
summarize(low,  "disagree-rate <= 50%")

# Correlate disagree RATE with features — also correlate the binary
# "is_high_disagree" flag, restricted to records where agree+disagree is not tiny
# (avoid records with n=1 artificially producing rate=0 or 1).
def corr_pair(xs, ys, name):
    mask = [(x,y) for x,y in zip(xs,ys) if x is not None]
    xs2 = [x for x,_ in mask]; ys2 = [y for _,y in mask]
    r, p = pearson(xs2, ys2)
    return name, r, p, len(xs2)

print("\n--- Pearson correlations (all records with >=1 agree/disagree) ---")
print(f"  {'feature':<12} {'vs rate':>10} {'p':>10} {'vs high-flag':>14} {'p':>10}")
rate = [r["rate"] for r in recs]
flag = [1 if r["rate"] > 0.5 else 0 for r in recs]
for key, lbl in [("mw","mw"),("heavy","heavy"),("pmids","pmids"),
                 ("syn","syn"),("evid","evid"),("nqa","nqa")]:
    xs = [r[key] for r in recs]
    _, r1, p1, _ = corr_pair(xs, rate, lbl)
    _, r2, p2, _ = corr_pair(xs, flag, lbl)
    print(f"  {lbl:<12} {r1:>10.4f} {p1:>10.2e} {r2:>14.4f} {p2:>10.2e}")

# Same, restricted to n >= 5 (filters out records with only 1–2 QAs that
# trivially produce rate ∈ {0, 0.5, 1})
filt = [r for r in recs if r["n"] >= 5]
print(f"\n--- Pearson correlations (records with >=5 agree/disagree; n={len(filt):,}) ---")
print(f"  {'feature':<12} {'vs rate':>10} {'p':>10} {'vs high-flag':>14} {'p':>10}")
rate = [r["rate"] for r in filt]; flag = [1 if r["rate"] > 0.5 else 0 for r in filt]
for key, lbl in [("mw","mw"),("heavy","heavy"),("pmids","pmids"),
                 ("syn","syn"),("evid","evid"),("nqa","nqa")]:
    xs = [r[key] for r in filt]
    _, r1, p1, _ = corr_pair(xs, rate, lbl)
    _, r2, p2, _ = corr_pair(xs, flag, lbl)
    print(f"  {lbl:<12} {r1:>10.4f} {p1:>10.2e} {r2:>14.4f} {p2:>10.2e}")

# Split × high-flag contingency
print("\n--- High-disagree rate by split ---")
print(f"  {'split':<6} {'records':>8} {'high':>6} {'high%':>7}")
by_split = {}
for r in recs:
    by_split.setdefault(r["split"], [0,0])
    by_split[r["split"]][0] += 1
    if r["rate"] > 0.5: by_split[r["split"]][1] += 1
for s in ["train","val","test"]:
    tot, h = by_split.get(s,[0,0])
    pct = h/tot*100 if tot else 0
    print(f"  {s:<6} {tot:>8,} {h:>6,} {pct:>6.2f}%")

records total: 15,509
records with disagree-rate > 50%: 89 (0.57%)
records with disagree-rate <= 50%: 15,420

high-disagree stratification:
  range         count
  0.5<r<=0.60      61
  0.6<r<=0.70      19
  0.7<r<=0.80       8
  0.8<r<=0.90       1
  0.9<r<=1.01       0

Top 20 CIDs by disagree rate (min 5 agree+disagree QAs):
       cid split  name                                n    dis%       mw  heavy   evid  pmids
    167812 val    Curcumenol                          6   83.3%    234.2     17      5      1
     11222 train  18-Hydroxycorticosterone            7   71.4%    362.2     26      2      1
     91770 train  Imazapic                            7   71.4%    275.1     20      2      2
    114633 train  d-Glucal                            7   71.4%    146.1     10      1      1
    121846 train  Chromanol 293B                      7   71.4%    324.1     22      2      1
    127790 train  Duartin                             7   71.4%    332.1     24      1      1
    244898 v

  pmids           -0.1489   0.00e+00        -0.0281   4.68e-04
  syn             -0.1508   0.00e+00        -0.0285   3.85e-04
  evid            -0.1791   0.00e+00        -0.0368   4.52e-06
  nqa             -0.2114   0.00e+00        -0.0488   1.18e-09

--- High-disagree rate by split ---
  split   records   high   high%
  train    10,820     45   0.42%
  val       2,340     23   0.98%
  test      2,349     21   0.89%


Bottom line
The "CIDs that mostly disagree" cohort is small (89 records) and overwhelmingly characterized by sparse evidence (~1 PMID, ~2 sentences, ~7 QAs) rather than by molecular size. It is also enriched in val/test splits by ~2×, suggesting dataset curators deliberately put harder, less-documented molecules into the held-out sets.